# UC1 — PCB (Printed Circuit Board): 2. Training

Fine-tune the Cosmos AnomalyGen adapter modules on the PCB data.

**Two paths:**
1. **Use the released checkpoint (fast).** You already downloaded
   `checkpoints/nvidia/Cosmos-AnomalyGen-PCB-2B` in notebook 0 — skip to §2.3 and go straight to generation.
2. **Train it yourself (multi-hour).** Follow §2.1–§2.2 to reproduce the checkpoint.

> **Note.** Only the small adapter modules (anomaly embedding, mask encoder, adapter)
> are trained; the diffusion backbone, tokenizer and text encoder stay frozen. Full
> training still takes several hours on a single GPU, so run it from a **terminal**,
> not a notebook cell.

> **How commands run in this tutorial.** All pipeline steps run inside the
> `cosmos-predict2` conda environment. In a notebook cell we prefix shell
> commands with `conda run -n cosmos-predict2` (add `--live-stream` to stream
> logs live). If you prefer, open a JupyterLab **Terminal**, run
> `conda activate cosmos-predict2` once, and paste the same commands without
> the `conda run` prefix.
>
> If that environment does not exist yet, build it first with the top-level
> [tutorial/notebooks/0-setup-cuda128.ipynb](../../0-setup-cuda128.ipynb) — see
> the prerequisite note below.

## 2.0 Set the project root

In [ ]:
# Resolve the repository root (the folder containing pyproject.toml) and cd into it,
# so every relative path below (datasets/, checkpoints/, results/, scripts/) resolves.
import os
d = os.getcwd()
while d != "/" and not os.path.exists(os.path.join(d, "pyproject.toml")):
    d = os.path.dirname(d)
LOCAL_PROJECT_DIR = d
os.chdir(LOCAL_PROJECT_DIR)
# Pipeline scripts read the finetuned models & write outputs under the repo root.
os.environ.setdefault("IMAGINAIRE_OUTPUT_ROOT", "./results")
print("Project root:", LOCAL_PROJECT_DIR)

## 2.1 The training configuration

We ship a ready UC1 config at **`ag_configs/UC1_pcb_NVDINOV2_2B_512.yaml`** (derived from the released
checkpoint's own `ag_config.yaml`, with `dataset_dir` pointed at your local data).
Key sections:

- **`dataloader_train.dataset.dataset_dir`** — path to the prepared dataset (`datasets/UC1_pcb`).
- **`anomaly_types`** — the `[TEXTURE, TYPE]` pairs to train on (must appear in *both*
  `dataloader_train.dataset` and `model.config.ag_config.anomaly_embedding`):
  ```yaml
          - [IC, bridge]
          - [passive_component, excess_solder]
          - [passive_component, missing]
  ```
- **`trainer.max_iter` / `checkpoint.save_iter` / `trainer.validation_iter`** — training
  length, checkpoint cadence, and how often validation images + KPIs are logged.
- **`trainer.early_stop`** — optional early stopping on the `nn` KPI.

Inspect the full config:

In [ ]:
!cat ag_configs/UC1_pcb_NVDINOV2_2B_512.yaml

## 2.2 Launch training (reference — run in a terminal)

> **Prerequisite (self-training only).** Self-training runs validation, which needs a
> validation testcase — **run notebook 3 §3.2b first** to build it, otherwise training
> stops as soon as it starts. (Using the released checkpoint in §2.3 doesn't need this.)

```bash
conda activate cosmos-predict2
export IMAGINAIRE_OUTPUT_ROOT=./results
CUDA_VISIBLE_DEVICES=0 torchrun --nproc_per_node=1 --master_port=12341 \
    -m scripts.anomaly_gen.ag_train \
    --config=cosmos_predict2/configs/base/ag_config.py \
    --ag_config=ag_configs/UC1_pcb_NVDINOV2_2B_512.yaml \
    -- experiment=predict2_anomaly_gen_ddp_2b
```

Training results, checkpoints and logs are written under
`results/anomaly_gen/<group>/<name>/`. Validation runs every `validation_iter`
steps and writes inpainted images + a `valid_kpi.csv` under
`.../valid/<step>/`, so you can watch quality improve.

<font color="red">**This is the multi-hour step.** For a quick end-to-end run, skip it and use the
released checkpoint below.</font>

## 2.3 Use the released checkpoint

The checkpoint downloaded in notebook 0 lets you skip straight to generation.
Confirm it is valid and see which defects it supports:

In [ ]:
!conda run -n cosmos-predict2 python scripts/utilities/validate_checkpoint.py checkpoints/nvidia/Cosmos-AnomalyGen-PCB-2B --step 14000

## 2.4 (Optional) Visualize the validation curve

After your own training run, plot per-defect **NN / MNN / FID** across validation
steps to pick the best checkpoint:

```bash
conda run -n cosmos-predict2 python -m scripts.anomaly_gen.visualize \
    --root results/anomaly_gen/<group>/<name>/valid \
    --output_dir plots/ \
    --validation_iter <validation_iter> \
    --max_iter <max_iter> \
    --anomaly_types IC+bridge passive_component+excess_solder passive_component+missing
```

<font color="red">**Note.** NN (higher is better) is the primary KPI; FID (lower is better) measures
distribution similarity in the C-RADIOv3 feature space but does not guarantee better
downstream performance — treat it as a reference metric.</font>

## Next Step

Proceed to [3-auto-mask-placement.ipynb](./3-auto-mask-placement.ipynb).